# Similarity-aware Label Smoothing

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import numpy as np
from tqdm import tqdm
from dataset_utils import get_data_loaders
import pandas as pd
from models import CifarResNET18, CifarDenseNET121, CifarWideResNET2810
from metrics import top_label_ece, nll_loss
import random
import os
from scipy.stats import spearmanr, wilcoxon


In [ ]:
def upper_tri_vec(D):
    """Vectorize upper triangle (excluding diagonal)."""
    idx = np.triu_indices_from(D, k=1)
    return D[idx]

def geometry_spearman(D1, D2):
    v1 = upper_tri_vec(D1)
    v2 = upper_tri_vec(D2)
    rho, _ = spearmanr(v1, v2)
    return rho

def relative_l2_change(D1, D2):
    v1 = upper_tri_vec(D1)
    v2 = upper_tri_vec(D2)
    return np.linalg.norm(v1 - v2) / np.linalg.norm(v1)

def permutation_test_geometry(D1, D2, n_perm=10000):
    v1 = upper_tri_vec(D1)
    v2 = upper_tri_vec(D2)

    observed = np.linalg.norm(v1 - v2)
    diffs = []

    for _ in range(n_perm):
        perm = np.random.permutation(len(v1))
        diffs.append(np.linalg.norm(v1 - v2[perm]))

    diffs = np.array(diffs)
    p_value = (np.sum(diffs >= observed) + 1) / (n_perm + 1)

    return observed, diffs.mean(), diffs.std(), p_value



In [ ]:
path = f"scores/05_cifar100_latent_distances.xlsx"
D_05 = torch.tensor(pd.read_excel(io=path, sheet_name="centroid_distance").iloc[:, 1:].to_numpy(dtype=np.float32), dtype=torch.float32).to(device)

path = f"scores/1_cifar100_latent_distances.xlsx"
D_1 = torch.tensor(pd.read_excel(io=path, sheet_name="centroid_distance").iloc[:, 1:].to_numpy(dtype=np.float32), dtype=torch.float32).to(device)

path = f"scores/4_cifar100_latent_distances.xlsx"
D_4 = torch.tensor(pd.read_excel(io=path, sheet_name="centroid_distance").iloc[:, 1:].to_numpy(dtype=np.float32), dtype=torch.float32).to(device)


pairs = [
    ("0.5", "1", D_05, D_1),
    ("1", "4", D_1, D_4),
    ("0.5", "4", D_05, D_4),
]

for a, b, Da, Db in pairs:
    rho = geometry_spearman(Da, Db)
    delta = relative_l2_change(Da, Db)
    obs, mu, std, p = permutation_test_geometry(Da, Db)

    print(f"β {a} → {b}")
    print(f"  Spearman ρ      = {rho:.3f}")
    print(f"  Relative change = {delta:.3f}")
    print(f"  Permutation p   = {p:.4g}\n")


In [ ]:
path = f"scores/05_cifar100_latent_distances.xlsx"
dists = torch.tensor(pd.read_excel(io=path, sheet_name="centroid_distance").iloc[:, 1:].to_numpy(dtype=np.float32), dtype=torch.float32).to(device)

topk_values, topk_indices = torch.topk(-dists[[i for i in range(100)]], 6, dim=1)
[clas.tolist() for clas in topk_indices]


indices = [1, 2, 3]
for j in topk_indices:
    selected_classes = [train_loader.dataset.classes[i] for i in j]
    print(selected_classes)

## Hyperparams

In [ ]:
seed = 0
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [ ]:
dataset = "cifar100"
batch_size = 128
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lr = 0.1
epochs = 200
num_classes = int(dataset.removeprefix("cifar"))
epsilon = 0.01

## Training Utils

In [ ]:
def accuracy(model, loader, k = (1, 5)):
    model.eval()
    correct = {key:0 for key in k}
    total = 0

    maxk = max(k)

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)

            _, pred = outputs.topk(maxk, dim=1, largest=True, sorted=True)

            for key in k:
                correct[key] += (pred[:, :key] == y.view(-1, 1)).any(dim=1).sum().item()
            total += y.size(0)

    return {key: correct[key] / total for key in k}

### Label Smoothing

In [ ]:
path = f"scores/4_cifar100_latent_distances.xlsx"
dists = torch.tensor(pd.read_excel(io=path, sheet_name="centroid_distance").iloc[:, 1:].to_numpy(dtype=np.float32), dtype=torch.float32).to(device)

def uniform_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return ((num_classes * (1 - epsilon) - 1) * y_onehot + epsilon) / (num_classes - 1)

def scores(y, k=5, tau=1.2):
    class_dists = dists[y]

    mask = torch.eye(class_dists.shape[1], device=device)[y]
    class_dists = class_dists.masked_fill(mask.bool(), float('inf'))
    sims = F.softmax(-class_dists / tau, dim=1)
    sims.scatter_(1, y.unsqueeze(1), 0.0)

    topk_values, topk_indices = torch.topk(sims, k, dim=1)
    result = torch.zeros_like(sims)
    result.scatter_(1, topk_indices, topk_values)

    result = result / (result.sum(dim=1, keepdim=True))

    return result

def similarity_aware_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return (1 - epsilon) * y_onehot + epsilon * scores(y)


## Training Loop

In [62]:
def train(model, train_loader, test_loader, classwise_target, optimizer=None, scheduler=None, epochs=10):

    for epoch in range(epochs):
        model.train()
        running_loss = 0

        for x, y in tqdm(train_loader, leave=False):
            x, y = x.to(device), y.to(device)

            targets = classwise_target[y]

            optimizer.zero_grad()
            logits = model(x)
            loss = -(targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x.size(0)

        if scheduler is not None: scheduler.step()

        test_acc = accuracy(model, test_loader)
        print(f"Epoch {epoch + 1}/{epochs} | Loss: {running_loss/len(train_loader.dataset):.4f} | Test Acc: {test_acc[1]:.4f} | Top-5 Test Acc: {test_acc[5]:.4f}")


## RUN Experiments

In [ ]:
classwise_target = similarity_aware_smooth_labels(torch.arange(end=num_classes).to(device), num_classes=num_classes, epsilon=epsilon)
print(classwise_target)
print(classwise_target.shape)

# classwise_target = torch.eye(n=num_classes, device=device)
# print(classwise_target)
# print(classwise_target.shape)


In [ ]:
train_loader, test_loader = get_data_loaders(dataset=dataset)

In [63]:
model = CifarResNET18(num_classes=num_classes).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4, nesterov=True)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=200
)

train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    classwise_target=classwise_target,
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=epochs,
)

Epoch 1/200 | Loss: 3.7741 | Test Acc: 0.1665 | Top-5 Test Acc: 0.4373


Epoch 2/200 | Loss: 2.9584 | Test Acc: 0.2546 | Top-5 Test Acc: 0.5465


Epoch 3/200 | Loss: 2.3786 | Test Acc: 0.3673 | Top-5 Test Acc: 0.7014


Epoch 4/200 | Loss: 2.0319 | Test Acc: 0.4158 | Top-5 Test Acc: 0.7466


Epoch 5/200 | Loss: 1.8137 | Test Acc: 0.4052 | Top-5 Test Acc: 0.7143


...

Epoch 196/200 | Loss: 0.0869 | Test Acc: 0.7857 | Top-5 Test Acc: 0.9384


Epoch 197/200 | Loss: 0.0868 | Test Acc: 0.7846 | Top-5 Test Acc: 0.9397


Epoch 198/200 | Loss: 0.0866 | Test Acc: 0.7865 | Top-5 Test Acc: 0.9391


Epoch 199/200 | Loss: 0.0869 | Test Acc: 0.7865 | Top-5 Test Acc: 0.9387


Epoch 200/200 | Loss: 0.0868 | Test Acc: 0.7852 | Top-5 Test Acc: 0.9380


In [64]:
logits_list = []
labels_list = []

model.eval()
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)

        logits_list.append(logits)
        labels_list.append(y)

logits_all = torch.cat(logits_list, dim=0)
labels_all = torch.cat(labels_list, dim=0)
print()
print("ECE:", top_label_ece(logits_all, labels_all))
print("NLL:", nll_loss(logits_all, labels_all))
test_acc = accuracy(model, test_loader)
print(f"Top-1 Test Acc: {test_acc[1]:.4f} | Top-5 Test Acc: {test_acc[5]:.4f}")
print()



ECE: 0.03958364948630333
NLL: 0.8582821488380432
Top-1 Test Acc: 0.7852 | Top-5 Test Acc: 0.9380



In [67]:
PATH = f"vae4_{'0'+str(epsilon).removeprefix("0.")}_{seed}.pth"
torch.save(model.state_dict(), PATH)